In [11]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
candidate_paths = [
    repo_root,
    repo_root / "notebooks" / "models",
    repo_root.parent,
    repo_root.parent.parent,
]
for candidate in candidate_paths:
    if (candidate / "pong_engine.cpython-314-x86_64-linux-gnu.so").exists():
        repo_root = candidate
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    import pygame
    print("pygame ok")
except Exception as exc:
    raise RuntimeError(f"pygame is required: {exc}")

try:
    import pong_engine
    print("pong_engine ok from", repo_root)
except Exception as exc:
    raise RuntimeError(f"Could not import pong_engine: {exc}")


pygame ok
pong_engine ok from /home/munna-01/PycharmProjects/Reinforcement-Learning-Basics/notebooks/models


In [9]:
FIELD_WIDTH = 640
FIELD_HEIGHT = 480
BALL_RADIUS = 10
PLAYER_WIDTH = 10
PLAYER_HEIGHT = 100
PLAYER_OFFSET = 80
PLAYER_ACCEL = 2000.0
PLAYER_TOP_SPEED = 500.0
BALL_ACCEL = 1.05
BALL_VEL = 170.0

game = pong_engine.Game(
    ball_radius=BALL_RADIUS,
    player_width=PLAYER_WIDTH,
    player_height=PLAYER_HEIGHT,
    field_width=FIELD_WIDTH,
    field_height=FIELD_HEIGHT,
    player_offset=PLAYER_OFFSET,
    player_accel=PLAYER_ACCEL,
    player_top_speed=PLAYER_TOP_SPEED,
    ball_accel=BALL_ACCEL,
    ball_vel=BALL_VEL,
)

print('Game initialized. Player 1: W/S | Player 2: Up/Down')


Game initialized. Player 1: W/S | Player 2: Up/Down


In [10]:
pygame.init()
screen = pygame.display.set_mode((FIELD_WIDTH, FIELD_HEIGHT))
pygame.display.set_caption('Pong')
clock = pygame.time.Clock()

def get_move_from_keys(up_key, down_key):
    if up_key and not down_key:
        return pong_engine.Move.UP
    if down_key and not up_key:
        return pong_engine.Move.DOWN
    return pong_engine.Move.NONE

running = True
while running:
    delta_time = clock.tick(60) / 1000.0

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    pressed = pygame.key.get_pressed()
    left_move = get_move_from_keys(pressed[pygame.K_w], pressed[pygame.K_s])
    right_move = get_move_from_keys(pressed[pygame.K_UP], pressed[pygame.K_DOWN])

    result = game.step(left_move, right_move, delta_time)

    screen.fill((0, 0, 0))

    # field border
    pygame.draw.rect(screen, (255, 255, 255), pygame.Rect(0, 0, FIELD_WIDTH, FIELD_HEIGHT), 2)
    pygame.draw.line(screen, (255, 255, 255), (FIELD_WIDTH // 2, 0), (FIELD_WIDTH // 2, FIELD_HEIGHT), 2)

    # paddles
    left_paddle = pygame.Rect(
        int(result.player_one_x),
        int(result.player_one_y),
        PLAYER_WIDTH,
        PLAYER_HEIGHT,
    )
    right_paddle = pygame.Rect(
        int(result.player_two_x),
        int(result.player_two_y),
        PLAYER_WIDTH,
        PLAYER_HEIGHT,
    )
    pygame.draw.rect(screen, (255, 255, 255), left_paddle)
    pygame.draw.rect(screen, (255, 255, 255), right_paddle)

    # ball
    pygame.draw.circle(screen, (255, 255, 255), (int(result.ball_x), int(result.ball_y)), BALL_RADIUS)
    pygame.display.flip()

    if result.game_over:
        print(f'Game over! Player {result.winner} wins.')
        running = False

pygame.quit()


Game over! Player 1 wins.
